# rail3D design review

Module-by-module visualizations of the 3D upgrade for approval **before** full data generation and training.
Physics gates V1–V4 (solver equivalence vs the verbatim Face3D code, FFT vs conv2d propagation,
angular-spectrum cross-check, sanity checks) are automated in `tests_physics_3d.py` and were all PASS —
see `data/generated/verification_report.json`.

Run top-to-bottom (CPU-safe on the laptop; set `DEVICE='cuda:0'` for the field maps to go faster).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
import torch
from rail3d import config, viz_setup

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
config.ensure_dirs()
print('device:', DEVICE)

## 1. Physical setup
λ = 8 mm (37.5 GHz), 60×30 grid at dx = 4 mm (240×120 mm aperture), horn at 224 mm / 55° in the x–z plane,
crown → metasurface 160 mm, metasurface → detectors 160 mm. Check every dimension and orientation.

In [ ]:
viz_setup.setup_diagram(save_path=config.FIGURE_DIR / 'setup_diagram.png');

## 2. Geometry: cross-sections and swept meshes
Defect CSV loops are width-matched to the intact reference (same convention as the 2D pipeline) and
blended in along the rail axis by a class-specific envelope g(y):
crack 5–20 mm sharp-edged, dent 30–100 mm Gaussian, wear 120–300 mm near-uniform.

In [ ]:
viz_setup.cross_section_overlay_figure(save_path=config.FIGURE_DIR / 'cross_sections.png');

In [ ]:
viz_setup.mesh_review_figure(save_path=config.FIGURE_DIR / 'mesh_review.png');

## 3. Fields at the metasurface plane
psi0 = direct horn term (face-independent, computed once); psi1 = single bounce off the rail (dominant);
psi2 = double bounce (≈10⁻³ of psi1). The 55° incidence produces x-fringes; short defects (crack)
visibly modulate the field along y — the signature the 3D system exploits and a 2D simulation cannot see.

In [ ]:
viz_setup.field_maps_figure(save_path=config.FIGURE_DIR / 'field_maps.png', device=DEVICE);

## 4. Trainable propagation
The exact Rayleigh–Sommerfeld kernel of Face3D's verified `prop3d`, applied via FFT linear convolution
(machine-precision identical to conv2d, ~10³× faster). The angular-spectrum propagator is kept as an
independent cross-check.

In [ ]:
viz_setup.propagator_figure(save_path=config.FIGURE_DIR / 'propagators.png');

## 5. Metasurface parameterizations
`SLM2D`: idealized phase-only mask. `MetaUnitSoft`: pillar-width map through the Face3D meta-atom
library (per-pixel polynomial fits, **central 60×30 crop** of the 80×80 arrays — the red box; please
confirm this crop is acceptable). Width uses a sigmoid reparameterization so gradients never die at
the [1, 3.8] mm bounds.

In [ ]:
viz_setup.metaunit_figure(save_path=config.FIGURE_DIR / 'metaunit_library.png');

## 6. Detectors
18 windows (18.2×11.2 mm) on a 6×3 grid; centers are trainable through sigmoid-edged soft masks whose
edge softness anneals during training; evaluation always uses the hard binary windows. Variance-based
pruning (Face3D criterion) trims 18 → 8 during training.

In [ ]:
viz_setup.detector_figure(save_path=config.FIGURE_DIR / 'detectors.png');

## 7. Barcode space (untrained)
Detector powers for one sample per class through the untrained optics, and their relative-L2 gap vs
the intact barcode. Training pushes defect gaps beyond the 0.40 margin, keeps intact compact,
separates class centroids, and trains the tiny linear head for crack/dent/wear classification.

In [ ]:
viz_setup.barcode_figure(save_path=config.FIGURE_DIR / 'barcode_untrained.png', device=DEVICE);

## Approval checklist
- [ ] Setup dimensions/orientations correct (diagram §1)
- [ ] Defect envelopes per class reasonable (§2)
- [ ] Field structure plausible (§3)
- [ ] Library crop 60×30 acceptable (§5)
- [ ] Detector layout/sizes acceptable (§6)

After approval: `python generate_dataset_3d.py --profile laptop --smoke` (20/class smoke set),
then the full run via `generate_overnight_laptop.py`.